# 2. Data Preparation

Preprocess data and split into train/validation/test sets (70/15/15).

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

## Load Dataset

In [2]:
sales_data = pd.read_csv("../dataset/sales.csv")
print(f"Dataset shape: {sales_data.shape}")
print(f"\nColumns: {sales_data.columns.tolist()}")

Dataset shape: (640840, 10)

Columns: ['Unnamed: 0', 'store_ID', 'day_of_week', 'date', 'nb_customers_on_day', 'open', 'promotion', 'state_holiday', 'school_holiday', 'sales']


## Data Preprocessing

In [3]:
def preprocess_data(df, drop_col, target_col):
    """ Drops one column and converts values in target_col: '0', 'A', 'B', 'C' -> 1.0 """
    df = df.copy()

    # Drop the column
    df = df.drop(columns=[drop_col])

    # # Replace values with 1.0
    df[target_col] = df[target_col].replace({ '0': 0.0, 'a': 1.0, 'b': 1.0, 'c': 1.0 }).astype(int)

    return df


In [8]:
# Make a copy to avoid modifying the original
data = sales_data.copy()

# Drop unnecessary columns
data = preprocess_data(sales_data, "date", "state_holiday")
data = preprocess_data(data, "Unnamed: 0", "state_holiday")

# Considering the sales only when the shop is open
data = data[data["open"]==1]
data = preprocess_data(data, "open", "state_holiday")

print(f"Columns after dropping: {data.columns.tolist()}")
print(f"\nData shape after preprocessing: {data.shape}")
print(f"\nData types:")
print(data.dtypes)
print(f"\nPreprocessed data sample:")
display(data.head())

Columns after dropping: ['store_ID', 'day_of_week', 'nb_customers_on_day', 'promotion', 'state_holiday', 'school_holiday', 'sales']

Data shape after preprocessing: (532016, 7)

Data types:
store_ID               int64
day_of_week            int64
nb_customers_on_day    int64
promotion              int64
state_holiday          int64
school_holiday         int64
sales                  int64
dtype: object

Preprocessed data sample:


,store_ID,day_of_week,nb_customers_on_day,promotion,state_holiday,school_holiday,sales
0,366,4,517,0,0,0,4422
1,394,6,694,0,0,0,8297
2,807,4,970,1,0,0,9729
3,802,2,473,1,0,0,6513
4,726,4,1068,1,0,0,10882


## Separate Features and Target

In [10]:
X = data.drop(columns='sales')
y = data['sales']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

Features shape: (532016, 6)
Target shape: (532016,)


## Split Data into Train/Validation/Test Sets

In [11]:
# First split: 70% train, 30% temp (for val and test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

# Second split: Split temp 50/50 to get validation and test sets
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print(f"Training set:   {X_train.shape} - {len(y_train)} samples")
print(f"Validation set: {X_val.shape} - {len(y_val)} samples")
print(f"Test set:       {X_test.shape} - {len(y_test)} samples")
print(f"\nTotal samples: {len(y_train) + len(y_val) + len(y_test)}")
print(f"\nSplit ratio:")
print(f"  Train:  {len(y_train)/(len(y_train) + len(y_val) + len(y_test))*100:.1f}%")
print(f"  Val:    {len(y_val)/(len(y_train) + len(y_val) + len(y_test))*100:.1f}%")
print(f"  Test:   {len(y_test)/(len(y_train) + len(y_val) + len(y_test))*100:.1f}%")

Training set:   (372411, 6) - 372411 samples
Validation set: (79802, 6) - 79802 samples
Test set:       (79803, 6) - 79803 samples

Total samples: 532016

Split ratio:
  Train:  70.0%
  Val:    15.0%
  Test:   15.0%


## Data Summary Statistics by Split

In [12]:
print("Training Set Statistics:")
print(f"  Sales mean: {y_train.mean():.2f}, std: {y_train.std():.2f}")

print("\nValidation Set Statistics:")
print(f"  Sales mean: {y_val.mean():.2f}, std: {y_val.std():.2f}")

print("\nTest Set Statistics:")
print(f"  Sales mean: {y_test.mean():.2f}, std: {y_test.std():.2f}")

Training Set Statistics:
  Sales mean: 6957.20, std: 3104.70

Validation Set Statistics:
  Sales mean: 6966.86, std: 3117.66

Test Set Statistics:
  Sales mean: 6961.23, std: 3095.35


## Save Prepared Data for Next Notebook

Save the split datasets so they can be loaded in the model training notebook.

In [16]:
import pickle
import os

# Create a data folder if it doesn't exist
data_folder = "../dataset/processed"
os.makedirs(data_folder, exist_ok=True)

# Save the splits
data_dict = {
    'X_train': X_train, 'y_train': y_train,
    'X_val': X_val, 'y_val': y_val,
    'X_test': X_test, 'y_test': y_test
}

with open(f"{data_folder}/train_val_test_split.pkl", 'wb') as f:
    pickle.dump(data_dict, f)

print(f"✓ Data splits saved to {data_folder}/train_val_test_split.pkl")

✓ Data splits saved to ../dataset/processed/train_val_test_split.pkl
